# Notebook Anggota A — Preprocessing V3 (Tahap 2)

**Tujuan:** membuat ulang dataset operasional, indeks random/temporal split, dan statistik final dari empat RAW + tiga file hasil audit. Keputusan kelompok: fokus artikel berita/klaim faktual Indonesia; opini/edukasi yang tidak berisi klaim berita jelas dikecualikan.

**Perhatian:** angka V3 adalah keluaran dari protokol yang dikunci untuk analisis, bukan verifikasi kebenaran setiap klaim secara manual. Statistik token IndoBERT dihitung pada bagian akhir (wajib).

## 1. Unggah 7 input
Unggah 4 XLSX RAW dan 3 CSV audit manual dengan nama yang sama. Jangan mengunggah dataset kandidat lama V1/V2 untuk menggantikan RAW.

In [ ]:
%pip -q install rapidfuzz scikit-learn transformers
from google.colab import files
u=files.upload()
print('Jumlah input terunggah:',len(u))


## 2. Membuat cache RAW dari empat XLSX
Pembacaan menggunakan ZIP/XML bawaan Python, tanpa mengubah sel Excel. Cache `raw_rows.pkl` dipakai pada skrip V3.

In [ ]:
import zipfile,xml.etree.ElementTree as ET,glob,os,re,pickle,collections,time
N='{http://schemas.openxmlformats.org/spreadsheetml/2006/main}'
def read_xlsx(fn):
 with zipfile.ZipFile(fn) as z:
  strings=[]
  with z.open('xl/sharedStrings.xml') as f:
   for ev,e in ET.iterparse(f,events=('end',)):
    if e.tag==N+'si':strings.append(''.join(x.text or '' for x in e.iter(N+'t')));e.clear()
  rows=[]
  with z.open('xl/worksheets/sheet1.xml') as f:
   for ev,e in ET.iterparse(f,events=('end',)):
    if e.tag==N+'row':
     d={}
     for c in e.findall(N+'c'):
      col=re.match('[A-Z]+',c.get('r')).group();v=c.find(N+'v')
      if c.get('t')=='inlineStr':
       t=c.find(N+'is');val=''.join(t.itertext()) if t is not None else ''
      else:
       val=v.text if v is not None else ''
       if c.get('t')=='s' and val:val=strings[int(val)]
      d[col]=val
     rows.append(d);e.clear()
  header=rows.pop(0)
  return [{header.get(k,k):v for k,v in r.items()} for r in rows],list(header.values())
os.makedirs('work_v3', exist_ok=True)
allrows=[]
for fn in sorted(glob.glob('dataset_*.xlsx')):
 t=time.time();r,h=read_xlsx(fn);print(os.path.basename(fn),len(r),h,'sec',round(time.time()-t,2),flush=True)
 for x in r:x['_sourcefile']=os.path.basename(fn)
 allrows.extend(r)
with open('work_v3/raw_rows.pkl','wb') as f:pickle.dump(allrows,f,protocol=4)
print('total',len(allrows), 'first', {k:str(v)[:80] for k,v in allrows[0].items()})


## 3. Menjalankan protokol V3
Kode berikut memproses label, NARASI, fallback judul, artikel valid, deduplikasi exact/near/title, lalu membuat random dan temporal split. Waktu eksekusi beberapa menit.

Semua kriteria audit dan alasan pengecualian disimpan. Angka V3 hasil Colab harus sama dengan dokumen Word bila 7 input identik.

In [ ]:
"""Pipeline V3 - reproducible source cleaning, conservative label handling, and group-safe split.
Only stdlib + sklearn for splitting (no models trained). Reads raw_rows.pkl as generated from 4 XLSX.
"""
import csv, re, os, json, collections, datetime, unicodedata, hashlib, statistics, random, pickle, math, zlib, heapq, itertools
from pathlib import Path
from collections import Counter,defaultdict
from sklearn.model_selection import train_test_split
from rapidfuzz import fuzz
ROOT=Path.cwd(); W=ROOT/'work_v3'; OUT=ROOT/'HASIL_V3'; OUT.mkdir(exist_ok=True)
DATA=pickle.load(open(W/'raw_rows.pkl','rb'))
SRC={'dataset_cnn_10k.xlsx':'CNN Indonesia','dataset_kompas_4k.xlsx':'Kompas','dataset_tempo_6k.xlsx':'Tempo','dataset_turnbackhoax_10k.xlsx':'TurnBackHoax'}
MIN_WORDS=8; SEED=42
WORD_RE=re.compile(r"(?u)\b[\w]+(?:[-’'][\w]+)*\b")
POS={'SALAH','HOAX','HOAKS','FALSE','HASUT','FITNAH','DISINFORMASI','MISINFORMASI','SCAM'}
NEG={'BENAR','TRUE','KLARIFIKASI','EDUKASI','KOREKSI','BERITA','CEK FAKTA','CEKFAKTA','CLARIFICATION','ACARA','FAKTA','ISU','OPINI','ARTIKEL','PENGUMUMAN'}
TAGALT='|'.join(re.escape(x) for x in sorted(POS|NEG,key=len,reverse=True))
# tolerate malformed single bracket and combinations: [SALAH), SALAH], (HOAX+HASUT), [EDUKASI, HOAX]
RE_PREFIX=re.compile(r'^\s*(?:[\[(]\s*)?(?P<tags>'+TAGALT+r')\b(?:\s*(?:[/+,;]|\s+dan\s+)\s*(?:'+TAGALT+r')\b){0,5}\s*(?:[\])]\s*)?\s*(?:[:：\-–—]\s*)?',re.I)
RE_PARTS=re.compile(r'\s*(?:/|\+|,|;|\bdan\b)\s*',re.I)
RE_NARASI=re.compile(r'(?im)(?:^|\n|={2,})\s*(?:[\[(]\s*)?NARASI(?:\s+HASUT)?(?:\s*[\])])?\s*[:：]?\s*')
RE_END=re.compile(r'(?i)(?:[\[(]\s*(?:PENJELASAN|KLARIFIKASI|REFERENSI|KESIMPULAN|KATEGORI|FAKTA)\s*[\])]\s*:?|\b(?:PENJELASAN|KLARIFIKASI|REFERENSI|KESIMPULAN|KATEGORI|FAKTA|HASIL PERIKSA FAKTA)\s*[:：]|\bFAKTANYA\s*[,.:]|(?:^|\n)[ =-]{3,}\s*PENJELASAN\b)')
RE_URL=re.compile(r'(?i)(?:https?://|www\.)\S+')
RE_HTML=re.compile(r'<[^<>]{0,300}>')
RE_CNN=re.compile(r'(?i)^\s*(?:[A-ZÀ-ÿ][\w\s.-]{1,40},\s*)?CNN\s+Indonesia\s*(?:--|[-–—])\s*')
RE_KOMPAS=re.compile(r'(?i)^\s*(?:[A-ZÀ-ÿ][\w\s.-]{1,40},\s*)?KOMPAS\s*\.\s*com\s*(?:--|[-–—])\s*')
RE_TEMPO=re.compile(r'(?i)^\s*(?:TEMPO\s*\.\s*CO|TEMPO)\s*,?\s*(?:[A-ZÀ-ÿ][\w\s.-]{1,35},?\s*)?(?:--|[-–—])\s*')
RE_SOURCE_START=re.compile(r'(?i)(?:\bCNN\s+Indonesia\s*(?:--|[–—])|\bKOMPAS\s*\.\s*com\s*(?:--|[-–—])|\bTEMPO\s*\.\s*CO\s*,?\s*Jakarta\s*[-–—])')
RE_BOILER=re.compile(r'(?i)\b(?:ADVERTISEMENT|SCROLL TO RESUME CONTENT|LEBIH BANYAK DARI KOLUMNIS)\b')
RE_GAMBAS=re.compile(r'(?i)\[Gambas:[^\]]{1,150}\]')
RE_LABEL_LEAK=re.compile(r'(?i)^\s*[\[(]?\s*(?:SALAH|HOAKS|HOAX|FALSE|HASUT|FITNAH|MISINFORMASI|DISINFORMASI)(?:\s*[+/]\s*(?:HOAX|HOAKS|HASUT|FITNAH|DISINFORMASI))?\s*[\])]?(?:\s*:|\s*[-–—])')
RE_FACTCHECK_VALID=re.compile(r'(?i)(?:^\s*[\[(](?:HOAKS|HOAX|SALAH|CEK FAKTA|CEKFAKTA|FAKTA)\b|^\s*(?:CEK FAKTA|CEKFAKTA|FAKTA ATAU HOAKS)\b|berdasarkan verifikasi kompas\s*\.\s*com sejauh ini, informasi ini tidak benar|^\s*hasil periksa fakta)')
RE_OPINION=re.compile(r'(?i)(?:artikel ini merupakan opini pribadi|tidak mencerminkan pandangan redaksi|\blebih banyak dari kolumnis\b|^\s*(?:\[?opini\]?|\[?kolom\]?)\s*[:\-])')
RE_EDUC=re.compile(r'(?i)^\s*(?:\[?edukasi\]?|\[?tips\]?)\s*[:\-]')
RE_CHECKER=re.compile(r'(?i)(?:\[\s*PENJELASAN\s*\]|\bhasil periksa fakta\b|\bfaktanya\s*[,.:]|\bselengkapnya (?:ada |pada )?penjelasan\b)')
EN={'the','is','are','was','were','with','for','from','this','that','and','not','have','has','you','your','into','as','about','at','of','on','who','which','their','after','before','false','claim','government','news'}
ID={'yang','dan','dari','dengan','untuk','pada','adalah','sebagai','tidak','ini','itu','oleh','di','ke','akan','atau','telah','dalam','tersebut','bahwa','juga','saat','menjadi','mereka','karena','sebuah','pemerintah','berita','masyarakat','terkait','serta'}

def clean_piece(s,body=False):
 s=unicodedata.normalize('NFKC',str(s or '')).replace('\u00a0',' ').replace('\u00ad','')
 if body:
  for rex in [RE_CNN,RE_KOMPAS,RE_TEMPO]:s=rex.sub('',s,count=1)
 s=RE_GAMBAS.sub(' ',s)
 s=re.sub(r'(?:\s*=\s*){3,}', ' ', s)
 s=RE_BOILER.sub(' ',s)
 s=RE_URL.sub(' ',s)
 s=RE_HTML.sub(' ',s)
 s=re.sub(r'(?im)^\s*(?:LIHAT JUGA|BACA JUGA)\s*:\s*[^\n]{0,250}', ' ',s)
 return re.sub(r'\s+',' ',s).strip()

def tag_info(title):
 s=str(title or '').strip();m=RE_PREFIX.match(s)
 if not m: return [],s,'NO_TAG'
 raw=m.group('tags');tags=[k.strip().upper() for k in RE_PARTS.split(raw) if k.strip()]
 tags=[t for t in tags if t in POS|NEG]
 tail=s[m.end():].lstrip(' :\t-–—')
 if not tags:return [],s,'UNKNOWN_TAG'
 return tags,tail,'RECOGNIZED'

def extract_narasi(body):
 if not body or body=='#ERROR!':return ''
 for m in RE_NARASI.finditer(body):
  p=body[m.end():];end=RE_END.search(p)
  if end:p=p[:end.start()]
  p=p.strip(' \t\n:')
  cand=clean_piece(p)
  if len(WORD_RE.findall(cand))>=3 and not RE_CHECKER.search(cand[:140]):return p
 return ''

def year_parse(x):
 s=str(x or '').strip()
 if re.fullmatch(r'\d{4,5}(?:\.0)?',s):
  n=float(s)
  if 35000<=n<=60000:return (datetime.datetime(1899,12,30)+datetime.timedelta(days=n)).year
 m=re.search(r'(?<!\d)(20\d{2})(?!\d)',s)
 return int(m.group(1)) if m else None

def norm(s):return re.sub(r'\W+',' ',unicodedata.normalize('NFKC',s or '').casefold()).strip()

def english_warning(s):
 tokens=[t.casefold() for t in WORD_RE.findall(s)[:125]]
 if len(tokens)<8:return False
 e=sum(t in EN for t in tokens);i=sum(t in ID for t in tokens)
 return e>=5 and e>=i*2 and e/(e+i+0.1)>=0.7

def get_audit(filename,status):
 with open(ROOT/filename,encoding='utf-8-sig',newline='') as f:r=list(csv.DictReader(f))
 return {int(x['raw_id']):(str(x.get(status,'')).strip().upper(),str(x.get('catatan_manual_v2',x.get('catatan','')))) for x in r}
AUD1=get_audit('AUDIT_40_judul_fallback(CEK MANUAL).csv','status_manual')
AUD2=get_audit('AUDIT_40_judul_hoazx_dikeluarkan(CEK MANUAL).csv','keputusan_manual')
AUD3=get_audit('AUDIT_V2_40_KASUS_HOLD_UNTUK_DICEK(SUDAH CEK MANUAL).csv','keputusan_manual_v2')
AUDALL=defaultdict(list)
for nm,a in [('fallback',AUD1),('excluded',AUD2),('hold_v2',AUD3)]:
 for key,value in a.items():AUDALL[key].append((nm,*value))
print('Audit 3 files distinct rows',len(AUDALL),'overlap',sum(len(v)>1 for v in AUDALL.values()),flush=True)

rows=[]
for raw_id,x in enumerate(DATA):
 source=SRC[x['_sourcefile']];label=int(source=='TurnBackHoax')
 title=str(x.get('Title') or '').strip();body=str(x.get('FullText') or '').strip();year=year_parse(x.get('Timestamp'))
 tag,striptitle,prefixstatus=tag_info(title)
 reasons=[];mode='TITLE_BODY' if label==0 else 'UNSET';decision='KEEP';rawtext='';clean=''
 if not title:decision='DROP';reasons.append('EMPTY_TITLE')
 if label==0:
  rawtext=(title+' '+body).strip();clean=(clean_piece(title)+' '+clean_piece(body,body=True)).strip()
  if not body or body=='#ERROR!':decision='DROP';reasons.append('MISSING_BODY')
  if RE_FACTCHECK_VALID.search(title) or RE_FACTCHECK_VALID.search(body[:300]):decision='DROP';reasons.append('FACTCHECK_IN_VALID_SOURCE')
  if RE_OPINION.search(title) or RE_OPINION.search(body[:450]) or RE_OPINION.search(body[-300:]):decision='DROP';reasons.append('OPINION_NOT_NEWS')
  if RE_EDUC.search(title):decision='DROP';reasons.append('EDUCATIONAL_NOT_NEWS')
 else:
  neg=set(tag)&NEG;pos=set(tag)&POS
  if not tag:decision='DROP';reasons.append('UNVERIFIABLE_HOAX_LABEL')
  elif neg:decision='DROP';reasons.append('MIXED_OR_NONHOAX_TAG')
  elif not pos:decision='DROP';reasons.append('NOT_POSITIVE_TAG')
  extract=extract_narasi(body)
  if extract:
   mode='NARASI';rawtext=(title+' '+extract).strip();clean=(clean_piece(striptitle)+' '+clean_piece(extract)).strip()
  else:
   mode='TITLE_FALLBACK';rawtext=title;clean=clean_piece(striptitle)
   if not body or body=='#ERROR!':decision='DROP';reasons.append('MISSING_OR_BROKEN_HOAX_BODY')
  if RE_CHECKER.search(clean):decision='DROP';reasons.append('FACTCHECK_LEAK_REMAINING')
 if not clean:decision='DROP';reasons.append('EMPTY_CLEAN')
 wc=len(WORD_RE.findall(clean))
 if wc<MIN_WORDS:decision='DROP';reasons.append('TOO_SHORT_8_WORDS')
 if english_warning(clean):decision='DROP';reasons.append('LIKELY_ENGLISH_TEXT')
 if label==0 and RE_SOURCE_START.search(clean[:220]):decision='DROP';reasons.append('SOURCE_DATELINE_REMAINING')
 if RE_BOILER.search(clean):decision='DROP';reasons.append('BOILERPLATE_REMAINING')
 if label==1 and RE_LABEL_LEAK.search(clean[:160]):decision='DROP';reasons.append('LABEL_LEAK_REMAINING')
 for origin,manual,note in AUDALL.get(raw_id,[]):
  if manual in ('TIDAK LAYAK','TETAP DIKELUARKAN'):
   # Exception: a manual 'not eligible' for mechanical label residue only may be restored if regex fixed.
   if origin=='fallback' and raw_id==31137 and not RE_LABEL_LEAK.search(clean):
    reasons.append('AUDIT_LABEL_FIXED');continue
   if origin=='hold_v2' and source in ('CNN Indonesia','Kompas','Tempo') and raw_id in (13285,14377) and not RE_SOURCE_START.search(clean[:200]):
    reasons.append('AUDIT_DATELINE_FIXED');continue
   decision='DROP';reasons.append('MANUAL_REJECT_'+origin)
  elif manual in ('RAGU',):
   # A few cases were purely malformed prefix. Cleaned and trusted class could be rescued via deterministic check.
   if label==1 and raw_id in (29569,31321,23929,22590,31560,31223,31078) and tag and set(tag)<=POS and not RE_LABEL_LEAK.search(clean):
    reasons.append('AUDIT_UNCERTAIN_RESOLVED_LABEL_FORMAT');continue
   decision='DROP';reasons.append('UNRESOLVED_MANUAL_DOUBT')
  elif manual in ('LAYAK','PERLU DIPERTAHANKAN'):
   reasons.append('AUDITED_POTENTIALLY_VALID_'+origin)
 # Not all accepted positive manual labels override automatic safety rules.
 rows.append({'raw_id':raw_id,'source':source,'label':label,'year':year or '',
  'title':title,'url':str(x.get('Url') or ''),'text_raw':rawtext,'text_clean':clean,
  'extract_mode':mode,'word_count':wc,'label_prefix':'+'.join(tag),'decision':decision,
  'reasons':';'.join(reasons),'audit_notes':' | '.join(f'{nm}: {s}: {note[:80]}' for nm,s,note in AUDALL.get(raw_id,[]))})
print('Pass 1 KEEP',sum(r['decision']=='KEEP' for r in rows),flush=True)

# exact normalized duplicates: keep earliest year then lowest id; cross-label conflicts exclude all (avoid inconsistent ground truth)
seen=defaultdict(list)
for r in rows:
 if r['decision']=='KEEP':seen[norm(r['text_clean'])].append(r)
exact_groups=0;exact_removed=0;cross_conflict=0
for key,grp in seen.items():
 if len(grp)<2:continue
 exact_groups+=1
 if len({r['label'] for r in grp})>1:
  cross_conflict+=len(grp)
  for r in grp:r['decision']='DROP';r['reasons']+=';EXACT_DUPLICATE_CONFLICT_LABEL'
 else:
  grp.sort(key=lambda r:(int(r['year']) if r['year']!='' else 9999,r['raw_id']))
  for r in grp[1:]:r['decision']='DROP';r['reasons']+=';EXACT_DUPLICATE_OF_'+str(grp[0]['raw_id']);exact_removed+=1
print('Exact removed',exact_removed,'label conflict',cross_conflict,flush=True)

# scalable near-dup candidate gen: 4-gram word shingles + bottom-12 minhash, 4 LSH signatures of 3 minima.
# Similarity thresholds measured with exact shingle Jaccard and rapidfuzz, conservative to avoid deleting distinct stories.
def tokenize(s):return [t.casefold() for t in WORD_RE.findall(norm(s))]
def get_shingles(s):
 tok=tokenize(s)
 if len(tok)<5:return set(zlib.crc32(' '.join(tok).encode('utf-8')) for _ in (0,))
 return {zlib.crc32(' '.join(tok[i:i+4]).encode('utf-8')) for i in range(len(tok)-3)}
kept=[r for r in rows if r['decision']=='KEEP']; byid={r['raw_id']:r for r in kept}
shingles={};bucket=defaultdict(list);pairs=set();
for j,r in enumerate(kept):
 idx=r['raw_id'];ss=get_shingles(r['text_clean']);shingles[idx]=ss
 # bottom-k via nsmallest; XOR salts permits approximate independent permutations
 base=heapq.nsmallest(24,ss)
 # Use 3 min-hash signatures with stable offset per band + pair-bucketing
 for b in range(12):
  key=(b,tuple(base[b*2:b*2+2]))
  # small common tokens guard: skip giant buckets
  if len(key[1])<2:continue
  candidates=bucket[key]
  if len(candidates)<150:
   for other in candidates:pairs.add((min(idx,other),max(idx,other)))
  if len(candidates)<200:bucket[key].append(idx)
 # secondary simhash bucket with first few tokens insensitive to small edits
 if (j+1)%10000==0:print('LSH processed',j+1,'pairs',len(pairs),flush=True)
print('Candidate pairs',len(pairs),flush=True)
# consider near duplication with broad strict rules. Exclude only similarity >= 0.90 Jaccard / string >= 96 + size ratio.
parent={i:i for i in byid}
def find(i):
 while parent[i]!=i:
  parent[i]=parent[parent[i]];i=parent[i]
 return i
def union(a,b):
 a,b=find(a),find(b)
 if a!=b:parent[max(a,b)]=min(a,b)
verified=[];cross_near=[]
for a,b in sorted(pairs):
 sa,sb=shingles[a],shingles[b]
 if not sa or not sb:continue
 jac=len(sa&sb)/len(sa|sb)
 if jac < .82:continue
 ratio=min(byid[a]['word_count'],byid[b]['word_count'])/max(byid[a]['word_count'],byid[b]['word_count'])
 score=fuzz.ratio(norm(byid[a]['text_clean']),norm(byid[b]['text_clean']))/100
 if (jac>=.9 and ratio>=.80) or (jac>=.85 and score>=.96 and ratio>=.85):
  if byid[a]['label']!=byid[b]['label']:cross_near.append((a,b,round(jac,3),round(score,3)))
  else:union(a,b);verified.append((a,b,round(jac,3),round(score,3)))
clusters=defaultdict(list)
for i in byid:clusters[find(i)].append(i)
near_groups=[v for v in clusters.values() if len(v)>1]
near_removed=0
for grp in near_groups:
 sr=sorted(grp,key=lambda idx:(int(byid[idx]['year']) if byid[idx]['year']!='' else 9999,idx))
 for idx in sr[1:]:
  r=byid[idx];r['decision']='DROP';r['reasons']+=';NEAR_DUPLICATE_OF_'+str(sr[0]);near_removed+=1
for a,b,_,_ in cross_near:
 for idx in (a,b):
  r=byid[idx];r['decision']='DROP';r['reasons']+=';NEAR_DUPLICATE_LABEL_CONFLICT'
print('near groups',len(near_groups),'removed',near_removed,'cross conflicts',len(cross_near),flush=True)

# Same normalized title with different article bodies also creates potential train/test leakage.
# Remove later-title copies conservatively (same class) before making splits.
title_groups=defaultdict(list)
for r in rows:
 if r['decision']=='KEEP':title_groups[norm(r['title'])].append(r)
title_group_duplicates=0;title_duplicate_removed=0;title_cross_label_removed=0
for title,grp in title_groups.items():
 if len(grp)<2 or not title:continue
 title_group_duplicates+=1
 if len({r['label'] for r in grp})>1:
  for r in grp:
   r['decision']='DROP';r['reasons']+=';REPEATED_TITLE_LABEL_CONFLICT'
   title_cross_label_removed+=1
 else:
  grp.sort(key=lambda r:(int(r['year']) if r['year']!='' else 9999,r['raw_id']))
  for r in grp[1:]:
   r['decision']='DROP';r['reasons']+=';REPEATED_TITLE_OF_'+str(grp[0]['raw_id'])
   title_duplicate_removed+=1
print('identical normalized title groups',title_group_duplicates,'later rows removed',title_duplicate_removed,flush=True)

final=[r for r in rows if r['decision']=='KEEP'];excluded=[r for r in rows if r['decision']!='KEEP']
# Random stratification 80/10/10; temporal <=2020, 2021, >=2022 within <=2023. Missing/foreign outside temporal.
y=[r['label'] for r in final]
tr,rest=train_test_split(list(range(len(final))),test_size=.2,stratify=y,random_state=SEED)
va,te=train_test_split(rest,test_size=.5,stratify=[y[i] for i in rest],random_state=SEED)
rmap={i:'LATIH' for i in tr};rmap.update({i:'VALIDASI' for i in va});rmap.update({i:'UJI' for i in te})
for i,r in enumerate(final):
 r['random_split']=rmap[i]
 if isinstance(r['year'],int) and 2015<=r['year']<=2020:r['temporal_split']='LATIH'
 elif r['year']==2021:r['temporal_split']='VALIDASI'
 elif isinstance(r['year'],int) and 2022<=r['year']<=2023:r['temporal_split']='UJI'
 else:r['temporal_split']='TIDAK_DIIKUTKAN'
# temporal subset sources continuous coverage, not all are guaranteed both classes in each year.
counts_source_temporal=Counter((r['source'],r['temporal_split']) for r in final)
continuous_sources=[s for s in SRC.values() if all(counts_source_temporal[s,part]>0 for part in ['LATIH','VALIDASI','UJI'])]
for r in final:r['temporal_main']=r['temporal_split'] if r['source'] in continuous_sources else 'TIDAK_DIIKUTKAN'

# stats and outputs
raw_source=Counter(SRC[x['_sourcefile']] for x in DATA)
cls=Counter(r['label'] for r in final)
by_source=Counter(r['source'] for r in final)
by_src_status=Counter((r['source'],r['decision']) for r in rows)
reasons=Counter(q for r in excluded for q in r['reasons'].split(';') if q and not q.startswith(('EXACT_DUPLICATE_OF','NEAR_DUPLICATE_OF')))
rc=Counter((r['random_split'],r['label']) for r in final)
tc=Counter((r['temporal_split'],r['label']) for r in final)
tm=Counter((r['temporal_main'],r['label']) for r in final)
raw_byyear=Counter((SRC[x['_sourcefile']],year_parse(x.get('Timestamp')) or 'TIDAK_ADA') for x in DATA)
final_byyear=Counter((r['source'],r['year'] or 'TIDAK_ADA') for r in final)
median={str(l):statistics.median(r['word_count'] for r in final if r['label']==l) for l in (0,1)}
pct_over_words={str(l):round(100*sum(r['word_count']>256 for r in final if r['label']==l)/cls[l],2) for l in (0,1)}
mode=Counter(r['extract_mode'] for r in final if r['label']==1)
summary={'protocol':'V3_STRICT_2026-10-13','raw_total':len(DATA),'source_raw':dict(raw_source),
 'selected_total':len(final),'excluded_total':len(excluded),'class_selected':dict(cls),'source_selected':dict(by_source),
 'by_source_status':{str(k):v for k,v in by_src_status.items()},'reasons_nonexclusive':dict(reasons),
 'min_word_threshold':MIN_WORDS,'manual_audit_distinct':len(AUDALL),
 'exact_duplicate_groups':exact_groups,'exact_duplicate_removed':exact_removed,'exact_cross_label_removed':cross_conflict,
 'near_duplicate_candidate_pairs':len(pairs),'near_duplicate_pairs_verified':len(verified),'near_duplicate_groups':len(near_groups),
 'near_duplicate_removed':near_removed,'near_duplicate_cross_label_pairs':len(cross_near),
 'repeated_title_groups':title_group_duplicates,'repeated_title_removed':title_duplicate_removed,'repeated_title_conflicts_removed':title_cross_label_removed,
 'class_median_words':median,'class_pct_over_256_words_not_tokens':pct_over_words,
 'hoax_extraction_modes':dict(mode),'random_split':{str(k):v for k,v in rc.items()},
 'temporal_split':{str(k):v for k,v in tc.items()},'temporal_main_sources':continuous_sources,'temporal_main':{str(k):v for k,v in tm.items()},
 'raw_year_source':{str(k):v for k,v in raw_byyear.items()},'selected_year_source':{str(k):v for k,v in final_byyear.items()},
 'tokenizer_indobert_status':'NOT_COMPUTED_ENV_REQUIRES_HF_TOKENIZER',
 'limitations':['V3 strict cohort is a deterministic protocol proposal, not a manually verified guarantee of every label.',
 'Label and source are fully confounded: all class 1 from TurnBackHoax, class 0 from three media.',
 'Near-duplicate candidate search uses approximate bottom-k shingles; recall not formally proven, only detected clusters removed.',
 'Language heuristic and opinion filter are conservative but not fully validated for precision/recall.',
 'IndoBERT >256 token statistic must be computed in Colab with exact tokenizer before numeric finalization.',
 'Temporal split does not prove future-topic robustness independently of source/time shifts.']}

def write_csv(path, data, cols):
 with open(path,'w',encoding='utf-8-sig',newline='') as f:
  w=csv.DictWriter(f,fieldnames=cols,extrasaction='ignore');w.writeheader();w.writerows(data)
DATA_FIELDS=['raw_id','source','label','year','title','url','text_raw','text_clean','extract_mode','word_count','label_prefix','decision','reasons','audit_notes','random_split','temporal_split','temporal_main']
write_csv(OUT/'DATASET_V3_KANDIDAT_OPERASIONAL.csv',final,DATA_FIELDS)
write_csv(OUT/'DATA_V3_DIKELUARKAN_DENGAN_ALASAN.csv',excluded,DATA_FIELDS)
write_csv(OUT/'INDEKS_SPLIT_RANDOM_V3.csv',[{k:r[k] for k in ('raw_id','source','label','year','random_split')} for r in final],['raw_id','source','label','year','random_split'])
write_csv(OUT/'INDEKS_SPLIT_TEMPORAL_V3.csv',[{k:r[k] for k in ('raw_id','source','label','year','temporal_split','temporal_main')} for r in final],['raw_id','source','label','year','temporal_split','temporal_main'])
write_csv(OUT/'DUPLIKAT_MIRIP_TERDETEKSI_V3.csv',[{'raw_id_1':a,'raw_id_2':b,'jaccard':ja,'string_similarity':sc,'label_conflict':int(byid[a]['label']!=byid[b]['label'])} for a,b,ja,sc in verified+cross_near],['raw_id_1','raw_id_2','jaccard','string_similarity','label_conflict'])
example=[]
for src in SRC.values():
 examples=[r for r in final if r['source']==src]
 for r in random.Random(42).sample(examples,min(5,len(examples))):
  example.append({'raw_id':r['raw_id'],'source':src,'label':r['label'],'extract_mode':r['extract_mode'],'raw_excerpt':r['text_raw'][:450],'clean_excerpt':r['text_clean'][:450]})
write_csv(OUT/'CONTOH_SEBELUM_SESUDAH_V3.csv',example,list(example[0]))
with open(OUT/'RINGKASAN_METODOLOGI_V3.json','w',encoding='utf-8') as f:json.dump(summary,f,indent=2,ensure_ascii=False)
print(json.dumps({k:v for k,v in summary.items() if k not in ('reasons_nonexclusive','raw_year_source','selected_year_source','by_source_status')},ensure_ascii=False,indent=2),flush=True)


## 4. Ringkasan angka dan pemeriksaan integritas

In [ ]:
import json, csv, collections
s=json.load(open('HASIL_V3/RINGKASAN_METODOLOGI_V3.json'))
print('RAW:',s['raw_total'],'TERPILIH:',s['selected_total'],'KELUAR:',s['excluded_total'])
print('Kelas:',s['class_selected'],'Sumber:',s['source_selected'])
print('Random:',s['random_split'])
print('Temporal seluruh sumber:',s['temporal_split'])
print('Temporal sumber utama:',s['temporal_main'])
assert s['selected_total']==29189, 'Hasil berbeda; cek nama/isi data dan audit!'
assert s['class_selected']=={'0':20867,'1':8322}
print('OK - statistik V3 cocok.')


## 5. Wajib: hitung token IndoBERT asli
Sel ini menentukan persen teks yang **benar-benar** lebih dari 256 token, termasuk token spesial. Angka dalam kata dari BAB 1 bukan pengganti. Kamu dapat mengunggah `TOKEN_INDOBERT_V3.json` ke ChatGPT untuk menyelesaikan angka yang kosong dalam dokumen.

In [ ]:
# Finalisasi statistik panjang IndoBERT dari CSV V3 (wajib untuk BAB 1 final).
# Gunakan TOKENIZER yang sama dengan Anggota B: indobenchmark/indobert-base-p1
import csv, json, collections, os
from pathlib import Path
from transformers import AutoTokenizer
from tqdm.auto import tqdm

DATA_PATH = 'HASIL_V3/DATASET_V3_KANDIDAT_OPERASIONAL.csv'
assert Path(DATA_PATH).exists(), 'File CSV V3 belum ditemukan, cek folder hasil ektraksi.'
tokenizer = AutoTokenizer.from_pretrained('indobenchmark/indobert-base-p1', use_fast=True)
total = collections.Counter(); over256 = collections.Counter(); lengths=collections.defaultdict(list)

with open(DATA_PATH, encoding='utf-8-sig', newline='') as f:
    reader = csv.DictReader(f)
    batch=[]
    for row in tqdm(reader, total=29189):
        batch.append((row['label'], row['text_clean']))
        if len(batch)==64:
            lengths_batch=tokenizer([s for _,s in batch], truncation=False, add_special_tokens=True, return_length=True)['length']
            for (label,_),ln in zip(batch,lengths_batch):
                total[label]+=1; over256[label]+=int(ln>256); lengths[label].append(ln)
            batch=[]
    if batch:
        lengths_batch=tokenizer([s for _,s in batch], truncation=False, add_special_tokens=True, return_length=True)['length']
        for (label,_),ln in zip(batch,lengths_batch):
            total[label]+=1; over256[label]+=int(ln>256); lengths[label].append(ln)

assert total['0']==20867 and total['1']==8322, f'Jumlah berbeda dari V3: {total}'
import statistics
res={
  'model_tokenizer': 'indobenchmark/indobert-base-p1',
  'token_count_includes_special_tokens': True,
  'threshold': 256,
  'class_0_VALID': {'n':total['0'],'count_over_256':over256['0'], 'percent_over_256':round(100*over256['0']/total['0'],2),'median_token':statistics.median(lengths['0'])},
  'class_1_HOAKS': {'n':total['1'],'count_over_256':over256['1'], 'percent_over_256':round(100*over256['1']/total['1'],2),'median_token':statistics.median(lengths['1'])}
}
Path('TOKEN_INDOBERT_V3.json').write_text(json.dumps(res,ensure_ascii=False,indent=2),encoding='utf-8')
print(json.dumps(res,ensure_ascii=False,indent=2))


## 6. Unduh seluruh keluaran
Unduh ZIP hasil V3 dan JSON tokenizer. Berkas CSV `DATASET_V3_KANDIDAT_OPERASIONAL.csv` berukuran besar, sehingga disarankan memakai ZIP.

In [ ]:
import zipfile
from pathlib import Path
from google.colab import files
with zipfile.ZipFile('PAKET_HASIL_ANGGOTA_A_V3.zip','w',compression=zipfile.ZIP_DEFLATED,compresslevel=1) as z:
    for p in Path('HASIL_V3').glob('*'):
        if p.is_file():z.write(p,arcname=p.name)
    if Path('TOKEN_INDOBERT_V3.json').exists(): z.write('TOKEN_INDOBERT_V3.json')
files.download('PAKET_HASIL_ANGGOTA_A_V3.zip')
if Path('TOKEN_INDOBERT_V3.json').exists():files.download('TOKEN_INDOBERT_V3.json')


## 7. Pemeriksaan tambahan eksternal (Mendeley)
Dataset eksternal Mendeley `600 news with valid hoax label.csv` perlu diunduh langsung dari https://data.mendeley.com/datasets/p3hfgr5j3m/1 lalu diperiksa jumlah kelas dan duplikasi terhadap dataset Kaggle. Jangan memasukkannya ke latih/validasi. Rekapnya dibutuhkan untuk deskripsi E6, tetapi pengujian model adalah tanggung jawab B/D.